# 11 — SQLite 测量读取器

IviumSoft 的 DataServer 将每次测量写入一个 SQLite 文件（`DataServer_*.idf.sqlite`），并维护
一个记录它们的目录（`index.sqlite`），默认位于 `C:\\IviumStat\\DataServer\\measurements`。
`Tools` 层以**只读**且 **WAL 安全**的方式读取这些文件，因此你甚至可以在 IviumSoft 仍在写入时
追踪一次测量 — 而且读取无需驱动或硬件。

### 你可以做什么

- 浏览目录并按序列号、技术或日期筛选
- 打开一次测量并读取其元数据、方法、分段和数据点
- 解码每个点的状态字节（过载、电流量程）
- 读取阻抗（EIS）点
- 增量追踪进行中的测量（`after_point_id` / `latest_point_id`）
- 导出为 CSV（若安装了 pandas，也可导出为 pandas DataFrame）

### 指向你的数据

本笔记本读取**你机器上真实的测量文件**。将下方的 `DATA_SERVER_DIR` 设置为你的 DataServer
测量文件夹。若该文件夹不存在，每个数据单元格都会自我保护，因此即使未安装 IviumSoft 你也能通读本笔记本。

In [ ]:
from pathlib import Path

from pyvium.tools import MeasurementReader, MeasurementIndex
print("pyvium.tools SQLite 读取器已导入")

## 1. 指向你的 DataServer

`index.sqlite` 位于测量文件夹中；每次测量的 `DataServer_*.idf.sqlite` 文件由其目录行
（`path` + `file`）定位，并在同一文件夹下解析。若你的安装不同，请修改 `DATA_SERVER_DIR`。

In [ ]:
# IviumSoft 的默认位置；请按你的机器修改。
DATA_SERVER_DIR = Path(r"C:\IviumStat\DataServer\measurements")
INDEX_PATH = DATA_SERVER_DIR / "index.sqlite"

HAVE_INDEX = INDEX_PATH.exists()
if HAVE_INDEX:
    print("使用目录:", INDEX_PATH)
else:
    print(f"在 {INDEX_PATH} 未找到 index.sqlite")
    print("请修改上方的 DATA_SERVER_DIR，使其指向你的 DataServer 测量文件夹。")

## 2. 浏览目录

`MeasurementIndex` 读取 `index.sqlite` 并按从新到旧返回条目。可按序列号（不区分大小写）、
技术、标题子串、项目/操作员以及日期范围筛选。

In [ ]:
entry = None
if HAVE_INDEX:
    with MeasurementIndex(str(INDEX_PATH)) as index:
        recent = index.entries(limit=10)
        for e in recent:
            print(f"  {e.start_time}  {str(e.technique):<20} 序列号={e.serialnumber}  {e.file}")

        # 筛选示例（取消注释并按需修改）：
        # index.entries(serialnumber="P33162")
        # index.entries(technique="CyclicVoltammetry")
        # index.entries(start_after="2024-01-01", start_before="2024-12-31")

    entry = recent[0] if recent else None
    print("\n已选择:", entry.file if entry else "目录为空")
else:
    print("已跳过 - 无目录（请在单元格 1 中设置 DATA_SERVER_DIR）")

## 3. 打开所选测量

`resolve_path(entry, base_dir)` 由目录行构建文件路径；`open_measurement(entry, base_dir)`
返回一个（尚未打开的）`MeasurementReader`。`base_dir` 即测量文件夹。

In [ ]:
MEASUREMENT_PATH = None
if entry is not None:
    resolved = MeasurementIndex.resolve_path(entry, str(DATA_SERVER_DIR))
    print("测量文件:", resolved)
    if Path(resolved).exists():
        MEASUREMENT_PATH = resolved
        with MeasurementReader(MEASUREMENT_PATH) as reader:
            print("数据库版本 :", reader.database_version)
            print("元数据     :", reader.metadata())
            print("测量       :", reader.measurements())
            print("方法参数   :", reader.method_parameters())
            for part in reader.measurement_parts()[:5]:
                print("  分段:", part)
    else:
        print("未找到文件 - 请调整 base_dir（resolve_path）以匹配你的目录结构。")
else:
    print("未选择测量")

## 4. 读取数据点（含解码状态）

`read_points()` 将每个点与其所属分段的上下文（循环 / 层级 / 通道）关联，并由 `statusbyte`
暴露出解码后的 `status`（过载标志与电流量程索引）。

In [ ]:
if MEASUREMENT_PATH:
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        points = reader.read_points()
    print(f"{len(points)} 个点；前 5 个:")
    for p in points[:5]:
        print(f"  id={p.point_id} t={p.t} x={p.x} y={p.y} z={p.z} "
              f"cycle={p.cycle} level={p.level} status={p.status}")
else:
    print("未选择测量")

## 5. 追踪进行中的测量

在 IviumSoft 仍在写入时，轮询 `latest_point_id()`，并将你已消费的最后一个 id 传给
`read_points(after_point_id=...)` 以仅获取新增点。读取器以只读且 WAL 安全的方式打开，因此
绝不会阻塞写入方。

In [ ]:
if MEASUREMENT_PATH:
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        latest = reader.latest_point_id()
        print("最新点 id:", latest)
        if latest and latest > 5:
            newer = reader.read_points(after_point_id=latest - 5)
            print(f"id {latest - 5} 之后的点:", [p.point_id for p in newer])
else:
    print("未选择测量")

## 6. 阻抗（EIS）点

对 EIS 技术，`pointfra` 表保存频率响应；`read_impedance()` 返回 `ImpedancePoint`。对非 EIS
测量（无 `pointfra`）它返回空列表，因此你无需事先知道技术类型。

In [ ]:
if MEASUREMENT_PATH:
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        eis = reader.read_impedance()
    if eis:
        for z in eis[:5]:
            print(f"  id={z.point_id}  f={z.frequency} Hz  Z'={z.z_re}  Z''={z.z_im}  "
                  f"quality={z.quality}")
    else:
        print("无阻抗点（并非 EIS 测量）")
else:
    print("未选择测量")

## 7. 导出

`to_csv()` 写出所有点；`to_dataframe()` 返回一个 pandas DataFrame（pandas 采用延迟导入，
因此仅在你调用时才需要）。

In [ ]:
if MEASUREMENT_PATH:
    import tempfile

    out_csv = Path(tempfile.gettempdir()) / "measurement_points.csv"
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        reader.to_csv(str(out_csv))
    print("已写入", out_csv)
    for line in out_csv.read_text(encoding="utf-8").splitlines()[:3]:
        print("  ", line)

    try:
        with MeasurementReader(MEASUREMENT_PATH) as reader:
            dataframe = reader.to_dataframe()
        print(dataframe.head())
    except ImportError:
        print("未安装 pandas - 跳过 to_dataframe()")
else:
    print("未选择测量")

## 8. 捷径：直接从 IviumSoft 获取最近一次测量

无需浏览目录，`Pyvium.get_db_file_name()` 直接返回**最近创建的**测量数据库的完整路径。该单元格
需要驱动已打开且 IviumSoft 正在运行（它是这里唯一与驱动通信的单元格）；读取器本身则不需要。

In [ ]:
from pyvium import Pyvium

try:
    Pyvium.open_driver()
    db_path = Pyvium.get_db_file_name()
    print("最近创建的数据库:", db_path)
    if db_path and Path(db_path).exists():
        with MeasurementReader(db_path) as reader:
            print("技术:", reader.method_parameters().get("Technique"))
            print("点数:", len(reader.read_points()))
except Exception as error:
    print(f"已跳过 ({type(error).__name__}: {error}) - 需要 IviumSoft 正在运行")
finally:
    try:
        Pyvium.close_driver()
    except Exception:
        pass

---

## 小结

| 任务 | API |
|------|-----|
| 打开目录 | `MeasurementIndex(index_path)`（默认文件夹 `C:\\IviumStat\\DataServer\\measurements`） |
| 浏览 / 筛选 | `.entries(serialnumber=..., technique=..., start_after=..., limit=...)` |
| 条目 -> 文件路径 | `MeasurementIndex.resolve_path(entry, base_dir)` |
| 条目 -> 读取器 | `.open_measurement(entry, base_dir)` |
| 打开一个测量文件 | `MeasurementReader(path)`（上下文管理器） |
| 元数据 / 方法 / 分段 | `.metadata()`, `.method_parameters()`, `.measurement_parts()` |
| 数据点（解码状态） | `.read_points()`, `point.status` |
| 追踪进行中的运行 | `.latest_point_id()`, `.read_points(after_point_id=...)` |
| 阻抗 | `.read_impedance()` |
| 导出 | `.to_csv(path)`, `.to_dataframe()` |
| 最近创建的数据库（需要 IviumSoft） | `Pyvium.get_db_file_name()` |

## 下一步

- **`10_instance_lifecycle_management.ipynb`** — 启动、接管并关闭 IviumSoft 实例
- **`07_data_processing.ipynb`** — 解析 IDF 文件并导出为 CSV